In [19]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
def data_generator(image_size = 224, training_batch = 8, validation_batch = 4):
  train_data_generator = ImageDataGenerator()
  valid_data_generator = ImageDataGenerator()
  train_generator = train_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/train/',
          target_size=(image_size, image_size),
          batch_size=training_batch,
          class_mode='categorical')

  validation_generator = valid_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/valid/',
          target_size=(image_size, image_size),
          batch_size=validation_batch,
          class_mode='categorical') 
  
  return train_generator, validation_generator

In [5]:
def build_model(hp):
    model = keras.Sequential()

    for i in range(hp.Int('num_layers', 3, 5)):
        model.add(
            layers.Conv2D(
                filters = hp.Int('conv_filter_' + str(i), min_value = 64, max_value = 128, step = 16),
                kernel_size = hp.Choice('conv_kernel_' + str(i), values = [3, 5]),
                activation = 'relu',
                input_shape = (224, 224, 3)
            )
        )
        model.add(layers.MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'))

    model.add(keras.layers.Flatten())

    for i in range(hp.Int('num_dense_layers', 2, 6)):
        model.add(
            keras.layers.Dense(
              units=hp.Int('dense_layer_units_' + str(i), min_value=256, max_value=1024, step=64),
              activation='relu'
              )
        )


    model.add(keras.layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()

    return model



In [6]:
train_generator, validation_generator = data_generator(training_batch=16, validation_batch=32)

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.


In [7]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials = 15,
    executions_per_trial = 1,
    directory = 'final',
    project_name = 'Tomato Disease'
    )


INFO:tensorflow:Reloading Oracle from existing project final\Tomato Disease\oracle.json
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 222, 222, 64)      1792      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 111, 111, 64)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 109, 109, 64)      36928     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 55, 55, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 53, 53, 64)        36928     
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 27, 27, 64)        0         
__________________________________

In [8]:
tuner.search_space_summary()

Search space summary
Default search space size: 19
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 3, 'max_value': 5, 'step': 1, 'sampling': None}
conv_filter_0 (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 128, 'step': 16, 'sampling': None}
conv_kernel_0 (Choice)
{'default': 3, 'conditions': [], 'values': [3, 5], 'ordered': True}
conv_filter_1 (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 128, 'step': 16, 'sampling': None}
conv_kernel_1 (Choice)
{'default': 3, 'conditions': [], 'values': [3, 5], 'ordered': True}
conv_filter_2 (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 128, 'step': 16, 'sampling': None}
conv_kernel_2 (Choice)
{'default': 3, 'conditions': [], 'values': [3, 5], 'ordered': True}
num_dense_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 6, 'step': 1, 'sampling': None}
dense_layer_units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 256,

In [ ]:
tuner.search(
    train_generator, 
    epochs = 10,
    steps_per_epoch = 1000,
    validation_steps = 20,
    validation_data = validation_generator        
)

In [9]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4,
 'conv_filter_0': 96,
 'conv_kernel_0': 3,
 'conv_filter_1': 64,
 'conv_kernel_1': 3,
 'conv_filter_2': 112,
 'conv_kernel_2': 5,
 'num_dense_layers': 5,
 'dense_layer_units_0': 704,
 'dense_layer_units_1': 320,
 'learning_rate': 0.0001,
 'conv_filter_3': 64,
 'conv_kernel_3': 5,
 'dense_layer_units_2': 704,
 'dense_layer_units_3': 320,
 'conv_filter_4': 80,
 'conv_kernel_4': 5,
 'dense_layer_units_4': 832,
 'dense_layer_units_5': 832}

In [10]:
tuner.results_summary()

Results summary
Results in final\Tomato Disease
Showing 10 best trials
Objective(name='val_accuracy', direction='max')
Trial summary
Hyperparameters:
num_layers: 4
conv_filter_0: 96
conv_kernel_0: 3
conv_filter_1: 64
conv_kernel_1: 3
conv_filter_2: 112
conv_kernel_2: 5
num_dense_layers: 5
dense_layer_units_0: 704
dense_layer_units_1: 320
learning_rate: 0.0001
conv_filter_3: 64
conv_kernel_3: 5
dense_layer_units_2: 704
dense_layer_units_3: 320
conv_filter_4: 80
conv_kernel_4: 5
dense_layer_units_4: 832
dense_layer_units_5: 832
Score: 0.9375
Trial summary
Hyperparameters:
num_layers: 5
conv_filter_0: 128
conv_kernel_0: 3
conv_filter_1: 128
conv_kernel_1: 3
conv_filter_2: 96
conv_kernel_2: 3
num_dense_layers: 5
dense_layer_units_0: 960
dense_layer_units_1: 640
learning_rate: 0.0001
conv_filter_3: 64
conv_kernel_3: 3
dense_layer_units_2: 832
dense_layer_units_3: 384
conv_filter_4: 128
conv_kernel_4: 3
dense_layer_units_4: 832
Score: 0.9312499761581421
Trial summary
Hyperparameters:
num_lay

In [11]:
tuner.get_best_models()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 222, 222, 96)      2688      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 111, 111, 96)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 109, 109, 64)      55360     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 55, 55, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 51, 51, 112)       179312    
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 26, 26, 112)       0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 22, 22, 64)        1

# K-FOLD VALIDATION

### create csv for dataset

In [12]:
import os
import pandas as pd

DIR_PATH = 'dataset/train'
file_name, classes = [], []

for x in os.listdir(DIR_PATH):
    for y in os.listdir(os.path.join(DIR_PATH, x)):
        file_name.append(os.path.join(os.getcwd(), DIR_PATH, x, y))
        classes.append(x)
        
df = pd.DataFrame({"file": file_name, "class" : classes})
df.to_csv("dataset.csv")

### K-FOLD validation 

In [13]:
from sklearn.model_selection import StratifiedKFold

In [14]:
train_data = pd.read_csv('dataset.csv')
Y = train_data[['class']]
skf = StratifiedKFold(n_splits = 5, random_state = 7, shuffle = True) 

In [29]:
idg = ImageDataGenerator(width_shift_range=0.1,
                         height_shift_range=0.1,
                         zoom_range=0.3,
                         fill_mode='nearest',
                         horizontal_flip = True,
                         rescale=1./255)

In [30]:
def get_model_name(k):
    return 'K_fold_validation_'+str(k)+'.h5'

In [36]:
VALIDATION_ACCURACY = []
VALIDAITON_LOSS = []

save_dir = 'saved_models/'
fold_var = 1

for train_index, val_index in skf.split(np.zeros(16030),Y):
    training_data = train_data.iloc[train_index]
    validation_data = train_data.iloc[val_index]
    
    train_data_generator = idg.flow_from_dataframe(training_data, directory = DIR_PATH,
                               x_col = "file", y_col = "class", target_size=(224, 224), batch_size=8,
                               class_mode = "categorical", shuffle = True)
    valid_data_generator  = idg.flow_from_dataframe(validation_data, directory = DIR_PATH,
                            x_col = "file", y_col = "class", target_size=(224, 224), batch_size=16,
                            class_mode = "categorical", shuffle = True)

    
    model =  tuner.get_best_models(num_models=1)[0]

    model.compile(loss='categorical_crossentropy',
              optimizer= tf.optimizers.Adam(1e-4),
              metrics=['accuracy'])

    
    checkpoint = tf.keras.callbacks.ModelCheckpoint(save_dir + get_model_name(fold_var), 
                            monitor='val_accuracy', verbose=1, 
                            save_best_only=True, mode='max')
    callbacks_list = [checkpoint]
    
    history = model.fit(train_data_generator,
                epochs=5,
                callbacks=callbacks_list,
                steps_per_epoch = 1000,
                validation_steps = 20,
                validation_data=valid_data_generator)
  
    model.load_weights("saved_models/K_fold_validation_"+str(fold_var)+".h5")

    results = model.evaluate(valid_data_generator)
    results = dict(zip(model.metrics_names,results))

    VALIDATION_ACCURACY.append(results['accuracy'])
    VALIDAITON_LOSS.append(results['loss'])

    tf.keras.backend.clear_session()

    fold_var += 1

Found 12824 validated image filenames belonging to 10 classes.
Found 3206 validated image filenames belonging to 10 classes.
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 222, 222, 96)      2688      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 111, 111, 96)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 109, 109, 64)      55360     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 55, 55, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 51, 51, 112)       179312    
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 26, 26, 112)       0       

1000/1000 [==============================] - 243s 243ms/step - loss: 0.3528 - accuracy: 0.8741 - val_loss: 0.4911 - val_accuracy: 0.8344

Epoch 00002: val_accuracy did not improve from 0.90000
Epoch 3/5
1000/1000 [==============================] - 249s 249ms/step - loss: 0.3034 - accuracy: 0.8955 - val_loss: 0.3324 - val_accuracy: 0.8969

Epoch 00003: val_accuracy did not improve from 0.90000
Epoch 4/5
1000/1000 [==============================] - 258s 258ms/step - loss: 0.2731 - accuracy: 0.9056 - val_loss: 0.1832 - val_accuracy: 0.9344

Epoch 00004: val_accuracy improved from 0.90000 to 0.93437, saving model to saved_models\K_fold_validation_2.h5
Epoch 5/5
1000/1000 [==============================] - 263s 263ms/step - loss: 0.2257 - accuracy: 0.9201 - val_loss: 0.1918 - val_accuracy: 0.9281

Epoch 00005: val_accuracy did not improve from 0.93437
201/201 [==============================] - 84s 419ms/step - loss: 0.2613 - accuracy: 0.9086
Found 12824 validated image filenames belonging t

Epoch 1/5
1000/1000 [==============================] - 345s 344ms/step - loss: 0.6186 - accuracy: 0.7964 - val_loss: 0.4407 - val_accuracy: 0.8500

Epoch 00001: val_accuracy improved from -inf to 0.85000, saving model to saved_models\K_fold_validation_4.h5
Epoch 2/5
1000/1000 [==============================] - 341s 341ms/step - loss: 0.3641 - accuracy: 0.8700 - val_loss: 0.2861 - val_accuracy: 0.9062

Epoch 00002: val_accuracy improved from 0.85000 to 0.90625, saving model to saved_models\K_fold_validation_4.h5
Epoch 3/5
1000/1000 [==============================] - 305s 305ms/step - loss: 0.3010 - accuracy: 0.8979 - val_loss: 0.2561 - val_accuracy: 0.9281

Epoch 00003: val_accuracy improved from 0.90625 to 0.92813, saving model to saved_models\K_fold_validation_4.h5
Epoch 4/5
1000/1000 [==============================] - 246s 246ms/step - loss: 0.2731 - accuracy: 0.9019 - val_loss: 0.2590 - val_accuracy: 0.9125

Epoch 00004: val_accuracy did not improve from 0.92813
Epoch 5/5
1000/1000 

# CNN

### Without preprocessing

In [96]:
def data_generator_without_preprocessing(image_size = 224, training_batch = 8, validation_batch = 4):
  train_data_generator = ImageDataGenerator()
  valid_data_generator = ImageDataGenerator()
  train_generator = train_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/train/',
          target_size=(image_size, image_size),
          batch_size=training_batch,
          class_mode='categorical')

  validation_generator = valid_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/valid/',
          target_size=(image_size, image_size),
          batch_size=validation_batch,
          class_mode='categorical') 
  
  return train_generator, validation_generator

In [99]:
def train_cnn_without_preprocessing(epochs = 10):
    time_callback = TimeHistory()
    train_generator, validation_generator = data_generator_without_preprocessing()
    model =  tuner.get_best_models(num_models=1)[0]
    model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
    r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
    )

    r.history['time'] = time_callback.times

    model.save('train_cnn_without_preprocessing.h5')
    pickle.dump(list(r.history.items()),open('train_cnn_without_preprocessing_score.h5', 'wb'))

In [100]:
train_cnn_without_preprocessing()

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 222, 222, 96)      2688      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 111, 111, 96)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 109, 109, 64)      55360     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 55, 55, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 51, 51, 112)       179312    
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 26, 26, 112)       0         
___________________________________

In [101]:
from sklearn.metrics import classification_report
_, valid_data = data_generator_without_preprocessing()
per = np.random.permutation(valid_data.n)
valid_data.index_array = per
model = load_model("train_cnn_without_preprocessing.h5")
print(classification_report(valid_data.classes[per], model.predict(valid_data).argmax(axis=1), target_names=list(os.listdir('dataset/train'))))

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.79      0.84      0.81       318
                        Tomato___Early_blight       0.65      0.74      0.69       360
                             Tomato___healthy       0.78      0.70      0.74       348
                         Tomato___Late_blight       0.88      0.70      0.78       354
                           Tomato___Leaf_Mold       0.56      0.66      0.61       337
                  Tomato___Septoria_leaf_spot       0.65      0.92      0.76       321
Tomato___Spider_mites Two-spotted_spider_mite       0.71      0.48      0.57       342
                         Tomato___Target_Spot       0.89      0.92      0.91       366
                 Tomato___Tomato_mosaic_virus       0.88      0.85      0.87       336
       Tomato___Tomato_Yellow_Leaf_Curl_Vi

### With preprocessing

In [109]:
def data_generator_with_preprocessing(image_size = 224, training_batch = 8, validation_batch = 4):
  train_data_generator = ImageDataGenerator(
  rotation_range=20,
          width_shift_range=0.2,
          height_shift_range=0.2,
          horizontal_flip=True
  )
  valid_data_generator = ImageDataGenerator()
  train_generator = train_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/train/',
          target_size=(image_size, image_size),
          batch_size=training_batch,
          class_mode='categorical')

  validation_generator = valid_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/valid/',
          target_size=(image_size, image_size),
          batch_size=validation_batch,
          class_mode='categorical') 
  
  return train_generator, validation_generator

In [110]:
def train_cnn_with_preprocessing(epochs = 10):
    time_callback = TimeHistory()
    train_generator, validation_generator = data_generator_with_preprocessing()
    model =  tuner.get_best_models(num_models=1)[0]
    model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
    r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
    )

    r.history['time'] = time_callback.times

    model.save('train_cnn_with_preprocessing.h5')
    pickle.dump(list(r.history.items()),open('train_cnn_with_preprocessing_score.h5', 'wb'))

In [111]:
train_cnn_with_preprocessing()

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 222, 222, 96)      2688      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 111, 111, 96)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 109, 109, 64)      55360     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 55, 55, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 51, 51, 112)       179312    
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 26, 26, 112)       0         
___________________________________

In [112]:
from sklearn.metrics import classification_report
_, valid_data = data_generator_without_preprocessing()
per = np.random.permutation(valid_data.n)
valid_data.index_array = per
model = load_model("train_cnn_with_preprocessing.h5")
print(classification_report(valid_data.classes[per], model.predict(valid_data).argmax(axis=1), target_names=list(os.listdir('dataset/train'))))

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.00      0.00      0.00       318
                        Tomato___Early_blight       0.00      0.00      0.00       360
                             Tomato___healthy       0.00      0.00      0.00       348
                         Tomato___Late_blight       0.00      0.00      0.00       354
                           Tomato___Leaf_Mold       0.00      0.00      0.00       337
                  Tomato___Septoria_leaf_spot       0.00      0.00      0.00       321
Tomato___Spider_mites Two-spotted_spider_mite       0.00      0.00      0.00       342
                         Tomato___Target_Spot       0.11      1.00      0.19       366
                 Tomato___Tomato_mosaic_virus       0.00      0.00      0.00       336
       Tomato___Tomato_Yellow_Leaf_Curl_Vi

C:\Users\harsh\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\metrics\_classification.py:1245: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\harsh\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\metrics\_classification.py:1245: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\harsh\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\metrics\_classification.py:1245: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

# Transfer of Learning

In [113]:
import os
import sys
import cv2
import time
import math
import shutil
import random
import pickle
import argparse
import numpy as np
from glob import glob
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import optimizers
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential, load_model
from tensorflow.python.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img
%matplotlib inline
sys.setrecursionlimit(1500)

In [114]:
class TimeHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.times = []

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.times.append(time.time() - self.epoch_time_start)


## VGG16

In [115]:
def data_generator(image_size = 224, training_batch = 8, validation_batch = 4):
  train_data_generator = ImageDataGenerator(
          preprocessing_function= tf.keras.applications.vgg16.preprocess_input,
          rotation_range=20,
          width_shift_range=0.2,
          height_shift_range=0.2,
          horizontal_flip=True
      )
  valid_data_generator = ImageDataGenerator(preprocessing_function=tf.keras.applications.vgg16.preprocess_input)
  train_generator = train_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/train/',
          target_size=(image_size, image_size),
          batch_size=training_batch,
          class_mode='categorical')

  validation_generator = valid_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/valid/',
          target_size=(image_size, image_size),
          batch_size=validation_batch,
          class_mode='categorical') 
  
  return train_generator, validation_generator

### Without pretrained weights

In [116]:
from tensorflow.keras.applications.vgg16 import VGG16
def VGG16Model_without_pretrained_weights(classes = 10, image_size = 224):
  model = Sequential()
  model.add(VGG16(include_top = False, pooling = 'avg', weights = None))
  model.add(Dense(classes, activation = 'softmax'))
  model.layers[0].trainable = False

  model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
  model.build(input_shape = (None, image_size, image_size, 3))
  model.summary()
  return model

In [117]:
def train_VGG16Model_without_pretrained_weights(epochs = 10):
  time_callback = TimeHistory()

  train_generator, validation_generator = data_generator()
  model = VGG16Model_without_pretrained_weights()
  r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
)
  
  r.history['time'] = time_callback.times

  model.save('VGG16Model_without_pretrained_weights.h5')
  pickle.dump(list(r.history.items()),open('VGG16Model_without_pretrained_weights_score.h5', 'wb'))

In [118]:
train_VGG16Model_without_pretrained_weights()

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
vgg16 (Functional)           (None, 512)               14714688  
_________________________________________________________________
dense_6 (Dense)              (None, 10)                5130      
Total params: 14,719,818
Trainable params: 5,130
Non-trainable params: 14,714,688
_________________________________________________________________
Epoch 1/10
1000/1000 [==============================] - 351s 347ms/step - loss: 2.2543 - accuracy: 0.1435 - specificity_at_sensitivity_1: 0.5317 - sensitivity_at_specificity_1: 0.5836 - auc_2: 0.5775 - auc_3: 0.5775 - val_loss: 2.2100 - val_accuracy: 0.1750 - val_specificity_at_sensitivity_1: 0.7111 - val_sensitivity_at_specificity_1: 0.7125 - val_auc_2: 0.6366 - val_auc_3: 0.6366
Epoch 2/10
1000/1000 [=======

In [119]:
from sklearn.metrics import classification_report
_, valid_data = data_generator()
per = np.random.permutation(valid_data.n)
valid_data.index_array = per
model = load_model("VGG16Model_without_pretrained_weights.h5")
print(classification_report(valid_data.classes[per], model.predict(valid_data).argmax(axis=1), target_names=list(os.listdir('dataset/train'))))


loss, accuracy, f1_score, precision, recall = model.evaluate(Xtest, ytest, verbose=0)

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.58      0.57      0.57       318
                        Tomato___Early_blight       0.43      0.55      0.49       360
                             Tomato___healthy       0.79      0.14      0.23       348
                         Tomato___Late_blight       0.62      0.26      0.36       354
                           Tomato___Leaf_Mold       0.89      0.02      0.05       337
                  Tomato___Septoria_leaf_spot       0.34      0.47      0.39       321
Tomato___Spider_mites Two-spotted_spider_mite       0.06      0.00      0.01       342
                         Tomato___Target_Spot       0.41      0.58      0.48       366
                 Tomato___Tomato_mosaic_virus       0.43      0.30      0.35       336
       Tomato___Tomato_Yellow_Leaf_Curl_Vi

NameError: name 'Xtest' is not defined

### With pretrained weights

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16
def VGG16Model_with_pretrained_weights(classes = 10, image_size = 224):
  model = Sequential()
  model.add(VGG16(include_top = False, pooling = 'avg', weights = 'imagenet'))
  model.add(Dense(classes, activation = 'softmax'))
  model.layers[0].trainable = False

  model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
  model.build(input_shape = (None, image_size, image_size, 3))
  model.summary()
  return model

In [ ]:
def train_VGG16Model_with_pretrained_weights(epochs = 10):
  time_callback = TimeHistory()

  train_generator, validation_generator = data_generator()
  model = VGG16Model_with_pretrained_weights()
  r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
)
  
  r.history['time'] = time_callback.times

  model.save('VGG16Model_with_pretrained_weights.h5')
  pickle.dump(list(r.history.items()),open('VGG16Model_with_pretrained_weights_score.h5', 'wb'))

In [ ]:
train_VGG16Model_with_pretrained_weights()

In [ ]:
loss, accuracy, f1_score, precision, recall = model.evaluate(Xtest, ytest, verbose=0)

# ResNET model

In [75]:
def data_generator(image_size = 224, training_batch = 8, validation_batch = 4):
  train_data_generator = ImageDataGenerator(
          preprocessing_function= tf.keras.applications.resnet50.preprocess_input,
          rotation_range=20,
          width_shift_range=0.2,
          height_shift_range=0.2,
          horizontal_flip=True
      )
  valid_data_generator = ImageDataGenerator(preprocessing_function=tf.keras.applications.resnet50.preprocess_input)
  train_generator = train_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/train/',
          target_size=(image_size, image_size),
          batch_size=training_batch,
          class_mode='categorical')

  validation_generator = valid_data_generator.flow_from_directory(
          f'{os.getcwd()}/dataset/valid/',
          target_size=(image_size, image_size),
          batch_size=validation_batch,
          class_mode='categorical') 
  
  return train_generator, validation_generator

### ResNET model without pretrained 

In [83]:
from tensorflow.keras.applications.resnet50 import preprocess_input, ResNet50

def resnetModel_without_pretrained_weights(classes = 10, image_size = 224):
  model = Sequential()
  model.add(ResNet50(include_top = False, pooling = 'avg', weights = None))
  model.add(Dense(classes, activation = 'softmax'))
  model.layers[0].trainable = False

  model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
  model.build(input_shape = (None, image_size, image_size, 3))
  model.summary()
  return model

In [84]:
def train_resnetModel_without_pretrained_weights(epochs = 10):
  time_callback = TimeHistory()

  train_generator, validation_generator = data_generator()
  model = resnetModel_without_pretrained_weights()
  r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
)
  
  r.history['time'] = time_callback.times

  model.save('resnetModel_without_pretrained_weights.h5')
  pickle.dump(list(r.history.items()),open('resnetModel_without_pretrained_weights_score.h5', 'wb'))

In [85]:
train_resnetModel_without_pretrained_weights()

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
Model: "sequential_8"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
resnet50 (Functional)        (None, 2048)              23587712  
_________________________________________________________________
dense_7 (Dense)              (None, 10)                20490     
Total params: 23,608,202
Trainable params: 20,490
Non-trainable params: 23,587,712
_________________________________________________________________
Epoch 1/10
1000/1000 [==============================] - 232s 227ms/step - loss: 2.7745 - accuracy: 0.2424 - specificity_at_sensitivity_2: 0.7185 - sensitivity_at_specificity_2: 0.6880 - auc_4: 0.6507 - auc_5: 0.6507 - val_loss: 2.4647 - val_accuracy: 0.2375 - val_specificity_at_sensitivity_2: 0.8069 - val_sensitivity_at_specificity_2: 0.8000 - val_auc_4: 0.7241 - val_auc_5: 0.7241
Epoch 2/10
1000/1000 [======

C:\Users\harsh\AppData\Local\Programs\Python\Python38\lib\site-packages\tensorflow\python\keras\utils\generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


In [89]:
from sklearn.metrics import classification_report
_, valid_data = data_generator()
per = np.random.permutation(valid_data.n)
valid_data.index_array = per
model = load_model("resnetModel_without_pretrained_weights.h5")
print(classification_report(valid_data.classes[per], model.predict(valid_data).argmax(axis=1), target_names=list(os.listdir('dataset/train'))))

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.62      0.64      0.63       318
                        Tomato___Early_blight       1.00      0.01      0.01       360
                             Tomato___healthy       0.97      0.10      0.18       348
                         Tomato___Late_blight       0.78      0.34      0.48       354
                           Tomato___Leaf_Mold       0.30      0.79      0.43       337
                  Tomato___Septoria_leaf_spot       0.29      0.81      0.43       321
Tomato___Spider_mites Two-spotted_spider_mite       0.34      0.27      0.30       342
                         Tomato___Target_Spot       0.88      0.50      0.64       366
                 Tomato___Tomato_mosaic_virus       0.89      0.22      0.35       336
       Tomato___Tomato_Yellow_Leaf_Curl_Vi

### ResNET model with pretrained 

In [91]:
from tensorflow.keras.applications.resnet50 import preprocess_input, ResNet50

def resnetModel_with_pretrained_weights(classes = 10, image_size = 224):
  model = Sequential()
  model.add(ResNet50(include_top = False, pooling = 'avg', weights = 'imagenet'))
  model.add(Dense(classes, activation = 'softmax'))
  model.layers[0].trainable = False

  model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy', keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5), tf.keras.metrics.AUC(from_logits=True), tf.keras.metrics.AUC(from_logits=True, curve='ROC')])
  model.build(input_shape = (None, image_size, image_size, 3))
  model.summary()
  return model

In [92]:
def train_resnetModel_with_pretrained_weights(epochs = 10):
  time_callback = TimeHistory()

  train_generator, validation_generator = data_generator()
  model = resnetModel_with_pretrained_weights()
  r = model.fit(
        train_generator,
        steps_per_epoch = 1000,
        epochs = epochs,
        validation_data = validation_generator,
        validation_steps = 20,
        callbacks=[time_callback]
)
  
  r.history['time'] = time_callback.times

  model.save('resnetModel_with_pretrained_weights.h5')
  pickle.dump(list(r.history.items()),open('resnetModel_with_pretrained_weights_score.h5', 'wb'))

In [93]:
train_resnetModel_with_pretrained_weights()

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
Model: "sequential_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
resnet50 (Functional)        (None, 2048)              23587712  
_________________________________________________________________
dense_8 (Dense)              (None, 10)                20490     
Total params: 23,608,202
Trainable params: 20,490
Non-trainable params: 23,587,712
_________________________________________________________________
Epoch 1/10
1000/1000 [==============================] - 252s 247ms/step - loss: 0.6041 - accuracy: 0.8043 - specificity_at_sensitivity_3: 0.9987 - sensitivity_at_specificity_3: 0.9986 - auc_6: 0.9788 - auc_7: 0.9788 - val_loss: 0.4201 - val_accuracy: 0.8750 - val_specificity_at_sensitivity_3: 1.0000 - val_sensitivity_at_specificity_3: 1.0000 - val_auc_6: 0.9886 - val_auc_7: 0.9886
Epoch 2/10
1000/1000 [======

C:\Users\harsh\AppData\Local\Programs\Python\Python38\lib\site-packages\tensorflow\python\keras\utils\generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


In [94]:
from sklearn.metrics import classification_report
_, valid_data = data_generator()
per = np.random.permutation(valid_data.n)
valid_data.index_array = per
model = load_model("resnetModel_with_pretrained_weights.h5")
print(classification_report(valid_data.classes[per], model.predict(valid_data).argmax(axis=1), target_names=list(os.listdir('dataset/train'))))

Found 16030 images belonging to 10 classes.
Found 3445 images belonging to 10 classes.
                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.97      0.92      0.94       318
                        Tomato___Early_blight       0.98      0.78      0.87       360
                             Tomato___healthy       0.91      0.96      0.93       348
                         Tomato___Late_blight       0.99      0.95      0.97       354
                           Tomato___Leaf_Mold       0.97      0.92      0.94       337
                  Tomato___Septoria_leaf_spot       0.87      0.98      0.92       321
Tomato___Spider_mites Two-spotted_spider_mite       0.76      0.93      0.84       342
                         Tomato___Target_Spot       1.00      0.93      0.96       366
                 Tomato___Tomato_mosaic_virus       0.99      0.99      0.99       336
       Tomato___Tomato_Yellow_Leaf_Curl_Vi